<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Libreta de Notas y el Precipicio 🧗
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/06%20-%20Aprendizaje%20por%20Refuerzo/Para%20Dummies/01_Valor_y_QLearning_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 01 (Funciones de valor y Q-learning)](../01_Funciones_de_Valor_y_Q_Learning.ipynb). La pregunta: **¿cómo aprende un agente qué hacer en cada situación, si solo recibe puntos?** La respuesta clásica es **llevar una libreta de notas**: una nota por cada lugar y cada movimiento que dice *«qué tan bueno es esto a la larga»*. Lo vamos a entender con una historia: **caminar junto a un precipicio**.

Al terminar podrás explicar:
1. Qué es una **nota de valor** y cómo se **corrige** con la experiencia (¡aprendiendo de la nota del vecino!).
2. Cómo funciona el algoritmo **Q-learning**, y qué lo hace «**optimista**».
3. La diferencia entre el viajero **optimista** (Q-learning) y el **cauteloso** (SARSA) frente a un precipicio.
4. Por qué **probar cosas al azar de vez en cuando** cuesta caro justo donde hay peligro.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../01_Funciones_de_Valor_y_Q_Learning.ipynb). Antes: [RL y MDPs (Dummies)](00_RL_y_MDPs_Dummies.ipynb).

---
## 1. La libreta de notas 📓

Imagina que estás en una ciudad que no conoces y quieres llegar a un lugar. En una libreta apuntas, para **cada esquina** y **cada dirección** que puedes tomar, una **nota**: *«si estoy aquí y voy por allá, ¿qué tan bien me irá a la larga?»*. Al principio todas las notas valen **cero** (no sabes nada). Con cada paso que das, **corriges una nota**.

La clave es **cómo** la corriges. No esperas a llegar al final del viaje; en cuanto das **un paso**, usas lo que acabas de ver:

> **Nota nueva ≈ nota vieja + un pasito hacia «lo que gané ahora + la mejor nota de la esquina a la que llegué».**

Es como preguntarle a un vecino: *«desde tu esquina, ¿qué tan bien te va?»*. Si el vecino tiene una nota alta, **tu paso hacia allá también merece nota alta**. Poco a poco, las notas altas **se van contagiando hacia atrás**, desde el tesoro hasta la salida, hasta que la libreta indica en cada esquina **qué dirección conviene**. Esa libreta es lo que el cuaderno estándar llama la *tabla Q*, y esta forma de aprender se llama ***Q-learning***. El «pasito» es qué tanto confías en la experiencia nueva frente a lo que ya sabías (la **tasa de aprendizaje**).

---
## 2. El precipicio 🧗

Un mapa de 4 filas y 12 columnas. Sales de la **esquina inferior izquierda** (`S`) y debes llegar a la **esquina inferior derecha** (`G`). La fila de abajo, entre ambas, es un **precipicio** (`C`): si pisas una casilla ahí **caes, pierdes 100 puntos y vuelves a la salida**. Cada paso cuesta **1 punto** (para que busques el camino corto).

```
. . . . . . . . . . . .
. . . . . . . . . . . .
. . . . . . . . . . . .
S C C C C C C C C C C G
```

Hay **dos caminos** razonables: el **corto**, pegado al borde del precipicio (13 pasos), y el **largo y seguro**, alejado del borde. Como a veces el agente **se tropieza** (con 10% de probabilidad elige una dirección al azar, para seguir probando cosas), caminar pegado al borde es **arriesgado**. Entrenamos a **dos viajeros** con la misma libreta, pero distinta manera de corregir las notas:

- 🌟 **El optimista (Q-learning):** apunta en la nota *«lo mejor que podría pasar desde la esquina siguiente»*, **como si nunca fuera a tropezar**.
- 🛡️ **El cauteloso (SARSA):** apunta *«lo que realmente me va a pasar desde la esquina siguiente, contando con que a veces me tropiezo»*.

Veamos qué camino aprende cada uno y cuántas veces **cae al precipicio mientras entrena** (con 3 intentos distintos de cada uno).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FILAS, COLS = 4, 12
INICIO, META = (3, 0), (3, 11)
MOV = [(0, -1), (1, 0), (0, 1), (-1, 0)]                 # 0 = izquierda, 1 = abajo, 2 = derecha, 3 = arriba

def paso(f, c, a):
    """Un movimiento: (nueva casilla, puntos, ¿cayo?). Pisar el precipicio: -100 y vuelta a la salida."""
    f2 = min(max(f + MOV[a][0], 0), FILAS - 1); c2 = min(max(c + MOV[a][1], 0), COLS - 1)
    if f2 == FILAS - 1 and 1 <= c2 <= COLS - 2:
        return INICIO, -100.0, True
    return (f2, c2), -1.0, False

def entrenar(tipo, episodios=2000, semilla=0, tropiezo=0.1, pasito=0.1):
    """tipo = 'optimista' (Q-learning) o 'cauteloso' (SARSA). Devuelve la libreta de notas y cuantas veces cayo al precipicio."""
    rng = np.random.default_rng(semilla)
    notas = np.zeros((FILAS, COLS, 4)); caidas = 0
    def elegir(f, c):                                      # la mayoria de las veces la mejor nota; a veces, al azar
        if rng.random() < tropiezo: return int(rng.integers(4))
        return int(np.argmax(notas[f, c] + 1e-9 * rng.random(4)))
    for _ in range(episodios):
        (f, c), a = INICIO, elegir(*INICIO)
        for _ in range(500):
            (f2, c2), puntos, cayo = paso(f, c, a); caidas += cayo
            a2 = elegir(f2, c2)
            if (f2, c2) == META: futuro = 0.0
            elif tipo == "optimista": futuro = notas[f2, c2].max()          # lo MEJOR que podria pasar desde la casilla siguiente
            else: futuro = notas[f2, c2, a2]                                # lo que REALMENTE va a hacer desde la casilla siguiente
            notas[f, c, a] += pasito * (puntos + futuro - notas[f, c, a])   # un pasito hacia (puntos + futuro)
            (f, c), a = (f2, c2), a2
            if (f, c) == META: break
    return notas, caidas

def camino(notas):
    """Ruta que sigue el viajero cuando elige SIEMPRE la mejor nota (sin tropezar)."""
    pos, ruta = INICIO, [INICIO]
    for _ in range(60):
        pos, _, _ = paso(*pos, int(np.argmax(notas[pos[0], pos[1]])))
        ruta.append(pos)
        if pos == META or ruta.count(pos) > 1: break
    return ruta

resumen = {}
for tipo in ("optimista", "cauteloso"):
    libretas = [entrenar(tipo, semilla=k) for k in range(3)]
    resumen[tipo] = {"libreta": libretas[0][0], "caidas": np.mean([c for _, c in libretas]), "largos": [len(camino(n)) - 1 for n, _ in libretas]}
    print(f"{tipo:<10}: camino de {resumen[tipo]['largos']} pasos (en 3 intentos) | caídas medias al precipicio durante el entrenamiento: {resumen[tipo]['caidas']:.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axes[0]
mapa = np.zeros((FILAS, COLS)); mapa[3, 1:11] = 1
ax.imshow(mapa, cmap="Reds", alpha=0.5, vmin=0, vmax=1.5)
for tipo, color in [("optimista", "#2563eb"), ("cauteloso", "#dc2626")]:
    ruta = camino(resumen[tipo]["libreta"]); desplazamiento = 0.08 if tipo == "optimista" else -0.08
    ax.plot([c + desplazamiento for _, c in ruta], [f + desplazamiento for f, _ in ruta], color=color, lw=3.5, marker="o", ms=4, label=f"{tipo} ({len(ruta) - 1} pasos)")
ax.text(0, 3, "S", ha="center", va="center", fontweight="bold", fontsize=13); ax.text(11, 3, "G", ha="center", va="center", fontweight="bold", fontsize=13)
ax.text(5.5, 3, "PRECIPICIO", ha="center", va="center", color="#7f1d1d", fontweight="bold")
ax.set_xticks([]); ax.set_yticks([]); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=2); ax.set_title("El camino que cada viajero escoge al final", fontweight="bold", fontsize=10)
axes[1].bar(["Optimista\n(Q-learning)", "Cauteloso\n(SARSA)"], [resumen["optimista"]["caidas"], resumen["cauteloso"]["caidas"]], color=["#2563eb", "#dc2626"])
axes[1].set_ylabel("Caídas al precipicio durante el entrenamiento (media)"); axes[1].set_title("Cuántas veces se cayó mientras aprendía", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()

assert all(n == 13 for n in resumen["optimista"]["largos"])                       # el optimista aprende SIEMPRE el camino corto (13 pasos)
assert all(n > 13 for n in resumen["cauteloso"]["largos"])                        # el cauteloso da siempre un rodeo
assert resumen["optimista"]["caidas"] > 1.5 * resumen["cauteloso"]["caidas"]      # y el optimista se cae bastante mas mientras aprende

---
## 3. ¿Qué pasó? 🤔

Mira los resultados: el **optimista** aprendió el **camino más corto del mundo** (13 pasos, pegado al borde), pero **se cayó muchísimas veces mientras entrenaba**. El **cauteloso** aprendió un **rodeo más largo**, pero **casi nunca se cayó**.

La razón es la manera de corregir las notas:

| | 🌟 **Optimista (Q-learning)** | 🛡️ **Cauteloso (SARSA)** |
|---|---|---|
| **Qué apunta** | Lo **mejor que podría pasar** desde la siguiente esquina | Lo que **de verdad va a pasar**, contando con que a veces se tropieza |
| **Sobre el precipicio** | «La esquina del borde vale mucho: desde ahí se llega rápido» (**olvida** que puede tropezar) | «Cerca del borde me caigo de vez en cuando: esa esquina vale menos» |
| **Camino final** | **Corto y arriesgado** (el mejor *si no tropezara nunca*) | **Largo y seguro** (el mejor *teniendo en cuenta que tropieza*) |
| **Al entrenar** | Se cae más veces (porque **sí** tropieza mientras explora) | Se cae poco |

Ninguno está «equivocado»: **responden preguntas distintas**. Q-learning aprende cuál sería **la mejor estrategia sin tropiezos** (aunque mientras entrena sufra); SARSA aprende **la mejor estrategia con los tropiezos incluidos**. Si el agente dejara de tropezar al final del entrenamiento (la probabilidad de probar al azar bajara a cero), ambos tenderían al **mismo camino**. En la vida real: si entrenas con tropiezos **baratos** (un simulador), el optimista es ideal; si cada tropiezo **cuesta caro** (un robot real junto a un precipicio de verdad), conviene el cauteloso.

> ⚠️ **Límite de este método:** una libreta con **una nota por cada esquina y movimiento** solo funciona si el mundo es **pequeño**. Para un videojuego, con millones de millones de situaciones posibles, la libreta no cabe: ese es el problema que resuelve el cuaderno 4 del módulo (*Deep RL*).

---
##### 🛠️ Práctica 1: Corrige una nota de la libreta (optimista y cauteloso)

Practica el «pasito» con números. Una esquina de tu libreta tiene la nota **2.0** para «ir a la derecha». Das ese paso, **ganas −1 punto** (cada paso cuesta uno) y llegas a una esquina cuyas notas son `[-5, -3, 4, -8]` (izquierda, abajo, derecha, arriba). Por un tropiezo, la acción que **de verdad** vas a tomar desde allí es la `0` (izquierda). El pasito es **0.1**. Recuerda: *nota nueva = nota vieja + pasito × (puntos + futuro − nota vieja)*.

1. **Optimista:** el `futuro` es la **mejor** nota de la esquina siguiente. Calcula `nota_nueva_opt`.
2. **Cauteloso:** el `futuro` es la nota de la acción que **realmente** tomará. Calcula `nota_nueva_cau`.
3. Repite la corrección del optimista **50 veces** con la misma experiencia, guardando el resultado en `nota_repetida`.
4. Comprueba que `nota_nueva_opt` es mayor que `nota_nueva_cau`, y que `nota_repetida` se acerca a `puntos + futuro_opt`.

In [ ]:
# Escribe tu código aquí

nota, puntos, pasito = 2.0, -1.0, 0.1
notas_siguiente = np.array([-5.0, -3.0, 4.0, -8.0])      # izquierda, abajo, derecha, arriba
accion_real = 0                                          # por un tropiezo, desde alli ira a la izquierda

# 1) Optimista: el futuro es la MEJOR nota de la esquina siguiente
# futuro_opt = notas_siguiente.max()
# nota_nueva_opt = nota + pasito * (puntos + futuro_opt - nota)

# 2) Cauteloso: el futuro es la nota de la accion que REALMENTE tomara
# futuro_cau = notas_siguiente[...]
# nota_nueva_cau = nota + pasito * (puntos + ... - nota)

# 3) Repetir la correccion del optimista 50 veces con la misma experiencia
# nota_repetida = nota
# for _ in range(50):
#     nota_repetida = nota_repetida + pasito * (puntos + futuro_opt - nota_repetida)

# 4) Comprobaciones con assert


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
nota, puntos, pasito = 2.0, -1.0, 0.1
notas_siguiente = np.array([-5.0, -3.0, 4.0, -8.0])      # izquierda, abajo, derecha, arriba
accion_real = 0                                          # por un tropiezo, desde alli ira a la izquierda

# 1) Optimista: el futuro es la MEJOR nota de la esquina siguiente
futuro_opt = notas_siguiente.max()
nota_nueva_opt = nota + pasito * (puntos + futuro_opt - nota)

# 2) Cauteloso: el futuro es la nota de la accion que REALMENTE tomara
futuro_cau = notas_siguiente[accion_real]
nota_nueva_cau = nota + pasito * (puntos + futuro_cau - nota)

print(f"Optimista: {nota} -> {nota_nueva_opt:.2f} | Cauteloso: {nota} -> {nota_nueva_cau:.2f}")

# 3) Repetir la correccion del optimista 50 veces con la misma experiencia
nota_repetida = nota
for _ in range(50):
    nota_repetida = nota_repetida + pasito * (puntos + futuro_opt - nota_repetida)
print(f"Tras 50 correcciones, la nota del optimista llega a {nota_repetida:.3f} (puntos + futuro = {puntos + futuro_opt})")

# 4) Comprobaciones
assert np.isclose(nota_nueva_opt, 2.1) and np.isclose(nota_nueva_cau, 1.2)
assert nota_nueva_opt > nota_nueva_cau
assert abs(nota_repetida - (puntos + futuro_opt)) < 0.05
print("La nota se corrige 'un pasito' hacia lo que ganaste + el futuro; el optimista sube la nota y el cauteloso la baja.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Un agente aprende apuntando en una libreta una *nota de valor* por cada situación y movimiento, corrigiéndola paso a paso con «lo que ganó ahora más la mejor nota de la casilla siguiente»; el viajero *optimista* (Q-learning) aprende el camino más corto aunque se caiga mucho al entrenar, y el *cauteloso* (SARSA) aprende el rodeo seguro porque cuenta con sus propios tropiezos.**

### Lo que aprendiste hoy:
- ✅ Una **nota de valor** dice qué tan bueno es algo **a la larga**, no solo ahora.
- ✅ Las notas se corrigen **paso a paso**, «contagiándose» desde el tesoro hacia la salida.
- ✅ **Q-learning** (optimista) aprende el mejor camino *sin tropiezos*; **SARSA** (cauteloso) el mejor *contando con ellos*.
- ✅ Explorar al azar **cuesta caro** donde hay peligro.
- ✅ La libreta solo funciona en mundos **pequeños**.

> 🎓 **Siguiente paso:** [La perilla de las probabilidades (Dummies)](02_Gradientes_de_Politica_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../01_Funciones_de_Valor_y_Q_Learning.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué en la libreta las notas altas «se contagian» desde el tesoro hacia la salida? ¿Qué crees que pasaría si el agente solo corrigiera sus notas al terminar el viaje y no en cada paso?
2. Vas a entrenar un robot repartidor que debe pasar junto a una escalera. En el simulador, caerse no cuesta nada; en el edificio real, un golpe daña el robot. ¿Con qué viajero (optimista o cauteloso) entrenarías en cada caso, y por qué?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>